# CliniqAI — Router Stage 1 (gatekeeper) v5 — fix false-positive regression from v4

**What happened:** v4 (full retrain, breast_histology positive source swapped to BreakHis) fixed
the original bug — real breast_histology images stopped being rejected as non_medical, and all 7
medical modalities route correctly. But after deploying v4, non-medical images started getting
accepted as `medical` and then routed by Stage 2 into `skin_lesion` (the modality closest to "a
generic color photo" among its 7 classes, so that's where a wrongly-accepted non-medical image
tends to land).

**Root cause:** v4 introduced automatic class-weighted loss, recomputed from whatever the current
train-split positive/negative counts happened to be. Swapping `breast_histology`'s source
(12,000-cap IDC patches -> 7,909 real BreakHis images) shifted the *overall* positive/negative
ratio as an unplanned side effect of a change that was only meant to touch one class. The
automatic weighting then amplified that shift into an asymmetric loss penalty that favored
predicting `medical` whenever the model was uncertain — matching exactly what was observed
(medical images still perfect, non-medical images increasingly misclassified as medical). Fixed
in this notebook by switching to plain unweighted loss (Step 11) — the underlying imbalance here
(~1.26x) never needed weighting, and unweighted loss can't be silently re-biased by a future
data-source change on any one class.

**Also added this time:** an explicit PASS/FAIL sanity check against *both* medical and
non-medical real-world images (Step 16), not just eyeballed probabilities — this exact
regression should have been visible in testing before deployment, and wasn't, because the
previous sanity check only printed raw predictions without checking them against an expected
label.

**Everything else matches v4** — same corrected BreakHis breast_histology source, same 6 Kaggle
negative sources, same wide-head EfficientNet-B0, same dedup-safe globbing.

Run top to bottom. No fresh downloads needed — everything is already on disk.

## Step 0 — Imports

In [1]:
import os
import sys
import json
import time
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


torch: 2.5.0+cu124 | cuda available: True


## Step 1 — Config

In [2]:
PROJECT_ROOT = Path(r"E:\programming\CliniqAI")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
NEGATIVES_DIR = DATA_RAW / "gatekeeper_negatives_v2"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models" / "router"
BACKEND_MODELS_DIR = PROJECT_ROOT / "backend" / "models" / "router"
SANITY_DIR = DATA_RAW / "manual_sanity_check"
# NEW in v5: separate folder for real non-medical images, expected to predict "non_medical".
# Drop a handful of everyday photos here -- especially any of the actual images that got
# misrouted to skin_lesion, for a direct regression test. Portraits/skin-close-ups are worth
# prioritizing since that's the specific failure mode observed.
SANITY_NEGATIVE_DIR = DATA_RAW / "manual_sanity_check_negative"
SANITY_NEGATIVE_DIR.mkdir(parents=True, exist_ok=True)

# modality -> its positive-source folder under data/raw/
POSITIVE_SOURCES = {
    "bone_fracture": DATA_RAW / "bone_fracture",
    "brain_mri": DATA_RAW / "brain_tumor",
    "breast_histology": DATA_RAW / "breast_histology_v2",   # <-- the only thing that changed vs. v2
    "kidney_ct": DATA_RAW / "kidney_stone",
    "pneumonia": DATA_RAW / "pneumonia",
    "retinal": DATA_RAW / "diabetic_retinopathy",
    "skin_lesion": DATA_RAW / "skin_lesion",
}
POSITIVE_CAP = 12000

# negative source subfolders under gatekeeper_negatives_v2/ -- 'fruits' is held out entirely,
# never enters train/val/test, only used as a post-hoc generalization check (Step 12)
NEGATIVE_SOURCES = ["food101", "scenes", "natural_images", "imagenet_mini", "human_faces", "synthetic_screenshots"]
HELD_OUT_SOURCE = "fruits"

IMG_SIZE = 224
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 30
SEED = 42
CHECKPOINT_NAME = "router_stage1_v5_efficientnet_b0"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTS = ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

MODELS_DIR.mkdir(parents=True, exist_ok=True)
BACKEND_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device)


device: cuda


## Step 2 — Glob helper (dedup-safe)

Same fix as the breast_histology specialist notebook: Windows globs are case-insensitive, so
matching several extension patterns and concatenating can double-count files. Dedupe by resolved
path everywhere.

In [3]:
def glob_images(folder, exts=IMG_EXTS):
    seen = set()
    out = []
    for ext in exts:
        for p in folder.glob(f"**/{ext}"):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp)
                out.append(p)
    return out


## Step 3 — Index negative sources

Reuses what's already on disk under `gatekeeper_negatives_v2/` — no fresh download. Prints
counts so you can sanity-check against the v2 numbers documented in the project log rather than
assuming they still match (folders may have changed since).

In [4]:
negative_records = []  # (path, "negative", source_tag)
for source in NEGATIVE_SOURCES:
    folder = NEGATIVES_DIR / source
    if not folder.exists():
        print(f"WARNING: {folder} not found, skipping {source}")
        continue
    imgs = glob_images(folder)
    for p in imgs:
        negative_records.append((str(p), "negative", source))
    print(f"{source}: {len(imgs)} images")

held_out_folder = NEGATIVES_DIR / HELD_OUT_SOURCE
held_out_images = glob_images(held_out_folder) if held_out_folder.exists() else []
print(f"\n{HELD_OUT_SOURCE} (held out, not used in train/val/test): {len(held_out_images)} images")
print(f"\nTotal negatives indexed: {len(negative_records)}")


food101: 101000 images
scenes: 24335 images
natural_images: 13798 images
imagenet_mini: 38668 images
human_faces: 7219 images
synthetic_screenshots: 6000 images

fruits (held out, not used in train/val/test): 335456 images

Total negatives indexed: 191020


## Step 4 — Index positive sources (7 modalities, capped 12,000 each)

`breast_histology` is the only source that changed vs. v2 — points at the corrected, deduped
BreakHis data instead of the old IDC 50x50 patches.

In [5]:
positive_records = []  # (path, "medical", modality_tag)
rng = random.Random(SEED)

for modality, folder in POSITIVE_SOURCES.items():
    if not folder.exists():
        print(f"WARNING: {folder} not found, skipping {modality}")
        continue
    imgs = glob_images(folder)
    if len(imgs) > POSITIVE_CAP:
        imgs = rng.sample(imgs, POSITIVE_CAP)
    for p in imgs:
        positive_records.append((str(p), "medical", modality))
    print(f"{modality}: {len(imgs)} images (source: {folder.name})")

print(f"\nTotal positives indexed: {len(positive_records)}")


bone_fracture: 10581 images (source: bone_fracture)
brain_mri: 7200 images (source: brain_tumor)
breast_histology: 7909 images (source: breast_histology_v2)
kidney_ct: 12000 images (source: kidney_stone)
pneumonia: 4751 images (source: pneumonia)
retinal: 3662 images (source: diabetic_retinopathy)
skin_lesion: 12000 images (source: skin_lesion)

Total positives indexed: 58103


## Step 5 — Corrupted-file validation pass (project convention)

In [6]:
all_records = negative_records + positive_records
bad_paths = set()
for path, *_ in all_records:
    try:
        with Image.open(path) as im:
            im.load()
    except Exception:
        bad_paths.add(path)

if bad_paths:
    print(f"Dropping {len(bad_paths)} corrupted/unreadable files.")
    all_records = [r for r in all_records if r[0] not in bad_paths]
else:
    print("No corrupted files found.")
print(f"{len(all_records)} usable images remain.")


No corrupted files found.
249123 usable images remain.


## Step 6 — Per-source train/val/test split (80/10/10)

Matches the v2 approach: split independently per source_tag so every source is proportionally
represented in all three sets (not patient-grouped here — these aren't per-patient medical
datasets like BreakHis, they're general image classification sources plus specialist folders).

In [7]:
by_source = defaultdict(list)
for r in all_records:
    by_source[r[2]].append(r)

train_records, val_records, test_records = [], [], []
for source, recs in by_source.items():
    recs = recs[:]
    random.Random(SEED).shuffle(recs)
    n = len(recs)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    train_records.extend(recs[:n_train])
    val_records.extend(recs[n_train:n_train + n_val])
    test_records.extend(recs[n_train + n_val:])

random.Random(SEED).shuffle(train_records)
random.Random(SEED).shuffle(val_records)
random.Random(SEED).shuffle(test_records)

print(f"train: {len(train_records)} | val: {len(val_records)} | test: {len(test_records)}")
print(f"\nPer-source counts (train/val/test):")
for source in sorted(by_source.keys()):
    tr = sum(1 for r in train_records if r[2] == source)
    va = sum(1 for r in val_records if r[2] == source)
    te = sum(1 for r in test_records if r[2] == source)
    print(f"  {source}: {tr} / {va} / {te}")


train: 199295 | val: 24908 | test: 24920

Per-source counts (train/val/test):
  bone_fracture: 8464 / 1058 / 1059
  brain_mri: 5760 / 720 / 720
  breast_histology: 6327 / 790 / 792
  food101: 80800 / 10100 / 10100
  human_faces: 5775 / 721 / 723
  imagenet_mini: 30934 / 3866 / 3868
  kidney_ct: 9600 / 1200 / 1200
  natural_images: 11038 / 1379 / 1381
  pneumonia: 3800 / 475 / 476
  retinal: 2929 / 366 / 367
  scenes: 19468 / 2433 / 2434
  skin_lesion: 9600 / 1200 / 1200
  synthetic_screenshots: 4800 / 600 / 600


## Step 7 — Write the standalone Dataset module (v5)

New file rather than overwriting `gatekeeper_dataset_v4.py` — that file documents the working
v4 fix and is worth preserving as-is.

In [8]:
dataset_module_path = SCRIPTS_DIR / "gatekeeper_dataset_v5.py"
dataset_module_code = '''"""
Standalone, importable Dataset for Router Stage 1 v5 training.
Must live in its own module (not a notebook cell) for Windows DataLoader multiprocessing
(num_workers > 0) to work -- worker subprocesses re-import this module fresh, so
ImageFile.LOAD_TRUNCATED_IMAGES must be set here at module level, not just in the notebook.
"""
import random
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class GatekeeperDatasetV5(Dataset):
    """records: list of (path, label_str, source_tag). label_str in {\'medical\',\'negative\'}."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _tag = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label


class RandomResolutionDegradation:
    """Randomly downsamples then upscales back. Applied only in the train transform.
    Kept as light insurance in v4 (p=0.3) -- not the primary fix, since the primary fix is
    the corrected breast_histology training data itself."""

    def __init__(self, p=0.3, low_res_range=(48, 128)):
        self.p = p
        self.low_res_range = low_res_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        low = random.randint(*self.low_res_range)
        img = img.resize((low, low), Image.BILINEAR)
        img = img.resize((w, h), Image.BILINEAR)
        return img
'''

dataset_module_path.write_text(dataset_module_code, encoding="utf-8")
print(f"Wrote {dataset_module_path}")

sys.path.insert(0, str(SCRIPTS_DIR))
from gatekeeper_dataset_v5 import GatekeeperDatasetV5, RandomResolutionDegradation


Wrote E:\programming\CliniqAI\scripts\gatekeeper_dataset_v5.py


## Step 8 — Labels, transforms

In [9]:
class_names = ["non_medical", "medical"]
label_to_idx = {"negative": 0, "medical": 1}

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    RandomResolutionDegradation(p=0.3, low_res_range=(48, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


## Step 9 — Datasets & DataLoaders

In [10]:
train_ds = GatekeeperDatasetV5(train_records, label_to_idx, transform=train_transform)
val_ds = GatekeeperDatasetV5(val_records, label_to_idx, transform=eval_transform)
test_ds = GatekeeperDatasetV5(test_records, label_to_idx, transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


train=199295 val=24908 test=24920


## Step 10 — Model (wide head, matches both router stages' existing checkpoint format)

In [11]:
def build_model(num_classes, hidden_dim=512):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, hidden_dim),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(hidden_dim, num_classes),
    )
    return model

model = build_model(len(class_names)).to(device, memory_format=torch.channels_last)
print(model.classifier)


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=512, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=512, out_features=2, bias=True)
)


## Step 11 — Loss, optimizer, scheduler, AMP

**v5 change:** v4 used automatic class-weighted loss, recomputed from whatever the current
positive/negative counts happened to be. Swapping breast_histology's source (12,000-cap IDC
patches -> 7,909 real BreakHis images) shifted the overall positive/negative ratio as a side
effect, which the automatic weighting then amplified into an asymmetric loss penalty — pushing
the model to prefer "medical" whenever uncertain, which is the likely cause of the false-positive
regression on non-medical images. The imbalance here (~1.26x) doesn't need weighting to begin
with. v5 uses plain unweighted loss so a future data-source swap on any one class can't silently
re-bias the whole binary decision again.

In [12]:
train_label_counts = Counter(r[1] for r in train_records)
print("train label counts:", dict(train_label_counts))

criterion = nn.CrossEntropyLoss()  # unweighted -- see note above
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)


train label counts: {'negative': 152815, 'medical': 46480}


## Step 12 — Training loop

Per-epoch checkpointing on val_loss improvement, resume cell below for kernel-restart recovery.

In [13]:
checkpoint_inprogress_path = MODELS_DIR / f"{CHECKPOINT_NAME}.checkpoint_inprogress.pt"
history = []
best_val_loss = float("inf")
epochs_no_improve = 0
start_epoch = 0

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {elapsed:.1f}s")

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "classes": class_names,
            "num_classes": len(class_names),
            "architecture": "efficientnet_b0",
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
        }, checkpoint_inprogress_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

print("Training done. Best val_loss:", best_val_loss)


epoch 01 | train_loss 0.0207 acc 0.9936 | val_loss 0.0081 acc 0.9978 | 2556.7s
epoch 02 | train_loss 0.0141 acc 0.9958 | val_loss 0.0091 acc 0.9970 | 2535.5s
epoch 03 | train_loss 0.0123 acc 0.9961 | val_loss 0.0064 acc 0.9981 | 2515.3s
epoch 04 | train_loss 0.0117 acc 0.9964 | val_loss 0.0065 acc 0.9974 | 2500.7s
epoch 05 | train_loss 0.0110 acc 0.9964 | val_loss 0.0062 acc 0.9977 | 2497.4s
epoch 06 | train_loss 0.0102 acc 0.9968 | val_loss 0.0043 acc 0.9985 | 2514.2s
epoch 07 | train_loss 0.0100 acc 0.9969 | val_loss 0.0046 acc 0.9988 | 2499.1s
epoch 08 | train_loss 0.0097 acc 0.9969 | val_loss 0.0079 acc 0.9974 | 2504.2s
epoch 09 | train_loss 0.0096 acc 0.9971 | val_loss 0.0086 acc 0.9968 | 2534.6s
epoch 10 | train_loss 0.0095 acc 0.9970 | val_loss 0.0051 acc 0.9985 | 2525.0s
epoch 11 | train_loss 0.0062 acc 0.9980 | val_loss 0.0032 acc 0.9991 | 2535.6s
epoch 12 | train_loss 0.0057 acc 0.9982 | val_loss 0.0048 acc 0.9985 | 2517.0s
epoch 13 | train_loss 0.0057 acc 0.9982 | val_loss 0

## Step 13 — Resume from checkpoint (only if the kernel was restarted mid-training)

Re-run Steps 0-11 first, then this cell instead of starting Step 12 from scratch.

In [14]:
# resume_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
# model.load_state_dict(resume_ckpt["model_state_dict"])
# start_epoch = resume_ckpt["epoch"]
# best_val_loss = resume_ckpt["best_val_loss"]
# print(f"Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
# # then re-run the Step 12 training loop cell, which will pick up from start_epoch


## Step 14 — Final save (project root + backend copy)

In [15]:
best_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

final_path = MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
torch.save(best_ckpt, final_path)

backend_final_path = BACKEND_MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
shutil.copy2(final_path, backend_final_path)

history_path = MODELS_DIR / f"{CHECKPOINT_NAME}.history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", backend_final_path)
print("Saved:", history_path)


C:\Users\anish\AppData\Local\Temp\ipykernel_14348\2688343157.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_ckpt = torch.load(checkpoint_inprogress_path, map_locat

Saved: E:\programming\CliniqAI\models\router\router_stage1_v5_efficientnet_b0.pt
Saved: E:\programming\CliniqAI\backend\models\router\router_stage1_v5_efficientnet_b0.pt
Saved: E:\programming\CliniqAI\models\router\router_stage1_v5_efficientnet_b0.history.json


## Step 15 — In-distribution test accuracy, per source

Reported for completeness, but per the project's own established finding (twice now, on the v3
attempts): this metric alone does NOT distinguish a real fix from a broken one, since every
source here is evaluated on the same distribution it trained on. Step 16 is the test that
actually matters.

In [16]:
model.eval()
all_preds, all_labels, all_sources = [], [], []
idx = 0
with torch.no_grad():
    for images, labels in test_loader:
        batch_size = images.size(0)
        images = images.to(device, memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_sources.extend(r[2] for r in test_records[idx:idx + batch_size])
        idx += batch_size

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
overall_acc = (all_preds == all_labels).mean()
print(f"Overall in-distribution test accuracy: {overall_acc:.4f} (n={len(all_labels)})")
print("\nPer-source accuracy:")
for source in sorted(set(all_sources)):
    sidx = [i for i, s in enumerate(all_sources) if s == source]
    acc = (all_preds[sidx] == all_labels[sidx]).mean()
    print(f"  {source}: {acc:.4f} (n={len(sidx)})")


Overall in-distribution test accuracy: 0.9992 (n=24920)

Per-source accuracy:
  bone_fracture: 0.9991 (n=1059)
  brain_mri: 1.0000 (n=720)
  breast_histology: 1.0000 (n=792)
  food101: 0.9997 (n=10100)
  human_faces: 0.9986 (n=723)
  imagenet_mini: 0.9974 (n=3868)
  kidney_ct: 1.0000 (n=1200)
  natural_images: 1.0000 (n=1381)
  pneumonia: 1.0000 (n=476)
  retinal: 1.0000 (n=367)
  scenes: 1.0000 (n=2434)
  skin_lesion: 0.9967 (n=1200)
  synthetic_screenshots: 1.0000 (n=600)


## Step 16 — Real-world sanity check, both directions, with explicit PASS/FAIL

This is the step that should have caught the v4 regression and didn't, because it only printed
raw probabilities without checking them against an expected label. Fixed here: every sanity
image gets an explicit expected label and an explicit PASS/FAIL, for both the medical side
(`manual_sanity_check/`, expected `medical`) and the non-medical side (NEW —
`manual_sanity_check_negative/`, expected `non_medical`). Don't deploy this checkpoint unless
both sides are 100% PASS, not just eyeballed as "looks about right".

In [17]:
def run_sanity_folder(folder, expected_class):
    images = glob_images(folder)
    results = []
    model.eval()
    for img_path in images:
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"{img_path.name}: failed to open ({e})")
            continue
        x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x), dim=1)[0]
        pred_idx = probs.argmax().item()
        pred_class = class_names[pred_idx]
        passed = pred_class == expected_class
        results.append((img_path.name, pred_class, probs[pred_idx].item(), passed))
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {img_path.name}: predicted {pred_class} ({probs[pred_idx].item():.4f}), expected {expected_class}")
    return results

print(f"=== Medical sanity images (expected: medical) — {SANITY_DIR} ===")
medical_results = run_sanity_folder(SANITY_DIR, "medical")

print(f"\n=== Non-medical sanity images (expected: non_medical) — {SANITY_NEGATIVE_DIR} ===")
if not glob_images(SANITY_NEGATIVE_DIR):
    print(f"No images found in {SANITY_NEGATIVE_DIR}. Drop a handful of real everyday photos "
          f"there before trusting this checkpoint -- ideally including the actual images that "
          f"got misrouted to skin_lesion in production, for a direct regression test.")
    negative_results = []
else:
    negative_results = run_sanity_folder(SANITY_NEGATIVE_DIR, "non_medical")

all_results = medical_results + negative_results
n_pass = sum(1 for r in all_results if r[3])
print(f"\n=== Summary: {n_pass}/{len(all_results)} passed ===")
if negative_results and all(r[3] for r in negative_results) and all(r[3] for r in medical_results):
    print("Both directions clean. This is the first checkpoint version worth deploying on this evidence alone.")
elif not negative_results:
    print("WARNING: no non-medical sanity images tested -- this is not enough evidence the false-positive regression is fixed.")


=== Medical sanity images (expected: medical) — E:\programming\CliniqAI\data\raw\manual_sanity_check ===
[PASS] Normal_breast_histology.jpg: predicted medical (1.0000), expected medical

=== Non-medical sanity images (expected: non_medical) — E:\programming\CliniqAI\data\raw\manual_sanity_check_negative ===
[PASS] data_batch_1_1263.png: predicted non_medical (0.9999), expected non_medical
[PASS] data_batch_1_292.png: predicted non_medical (0.9930), expected non_medical
[PASS] data_batch_3_9536.png: predicted non_medical (0.9999), expected non_medical
[PASS] apple.jpg: predicted non_medical (1.0000), expected non_medical
[PASS] bird.jpg: predicted non_medical (0.9999), expected non_medical

=== Summary: 6/6 passed ===
Both directions clean. This is the first checkpoint version worth deploying on this evidence alone.


## Step 17 — Generalization check on held-out `fruits` (never trained on)

Confirms the fix doesn't overcorrect into "everything is medical" — same discipline as v2's
original held-out generalization test. Complements Step 16's negative-sanity folder with a
larger, automatic sample.

In [18]:
if held_out_images:
    model.eval()
    correct = 0
    sample = held_out_images[:500] if len(held_out_images) > 500 else held_out_images
    with torch.no_grad():
        for img_path in sample:
            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                continue
            x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                probs = torch.softmax(model(x), dim=1)[0]
            pred = probs.argmax().item()
            if class_names[pred] == "non_medical":
                correct += 1
    print(f"fruits (held out): {correct}/{len(sample)} correctly classified non_medical ({correct/len(sample):.4f})")
else:
    print("No held-out fruits images found -- skipping generalization check.")


fruits (held out): 500/500 correctly classified non_medical (1.0000)


## Notes

- **Do not point `backend/app/config.py`'s `ROUTER_STAGE1_PATH` at v5 until Step 16 shows 100%
  PASS on both the medical and non-medical sanity folders**, and Step 17's fruits accuracy is
  back near v2's original baseline (not just "close" — this is exactly the metric that silently
  regressed between v2 and v4 without anyone checking it against a number, only eyeballed).
- If Step 16's non-medical side still fails after this fix: the class-weighting hypothesis was
  wrong, or only partially right, and the resolution-degradation augmentation (still present at
  p=0.3, unchanged from v4) becomes the next thing to isolate — try removing it entirely in a v6
  pass rather than changing it and the loss weighting at the same time, to keep the diagnosis
  clean.
- Worth committing `scripts/gatekeeper_dataset_v5.py`, this notebook, and populating
  `data/raw/manual_sanity_check_negative/` with real examples (including the actual production
  images that triggered this bug) to `feature/router` regardless of outcome.
- `models/router/router_stage1_v5_efficientnet_b0.pt` (and the backend copy) needs uploading to
  the shared Google Drive once validated, same as the other checkpoints.